In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import pandas as pd
train = pd.read_csv("/content/train_v9rqX0R.csv")
test  = pd.read_csv("/content/test_AbJTz2l.csv")

In [ ]:
print(train.head())
print(test.head())

  Item_Identifier  Item_Weight Item_Fat_Content  Item_Visibility  \
0           FDA15         9.30          Low Fat         0.016047   
1           DRC01         5.92          Regular         0.019278   
2           FDN15        17.50          Low Fat         0.016760   
3           FDX07        19.20          Regular         0.000000   
4           NCD19         8.93          Low Fat         0.000000   

               Item_Type  Item_MRP Outlet_Identifier  \
0                  Dairy  249.8092            OUT049   
1            Soft Drinks   48.2692            OUT018   
2                   Meat  141.6180            OUT049   
3  Fruits and Vegetables  182.0950            OUT010   
4              Household   53.8614            OUT013   

   Outlet_Establishment_Year Outlet_Size Outlet_Location_Type  \
0                       1999      Medium               Tier 1   
1                       2009      Medium               Tier 3   
2                       1999      Medium               Tier

In [ ]:
train["source"] = "train"
test["source"]  = "test"
data = pd.concat([train, test], ignore_index=True, sort=False)

print(f"Train: {train.shape}, Test: {test.shape}, Combined: {data.shape}")

Train: (8523, 13), Test: (5681, 12), Combined: (14204, 13)


In [ ]:
data.isnull().sum()

,0
Item_Identifier,0
Item_Weight,2439
Item_Fat_Content,0
Item_Visibility,0
Item_Type,0
Item_MRP,0
Outlet_Identifier,0
Outlet_Establishment_Year,0
Outlet_Size,4016
Outlet_Location_Type,0


In [ ]:
#Non-consumables shouldn't have a fat content label
non_consumable_prefixes = ["NC", "DR", "HH", "FD"]
# Actually only NC prefix items are non-food
data.loc[data["Item_Identifier"].str[:2] == "NC", "Item_Fat_Content"] = "Non-Edible"

In [ ]:
data["Item_Fat_Content"].unique()
# As this column have both LF & low fat . SSO we will convert it into same name

array(['Low Fat', 'Regular', 'Non-Edible', 'low fat', 'LF', 'reg'],
      dtype=object)

In [ ]:
# As this column have both LF & low fat. SSO we will convert it into same name
data["Item_Fat_Content"] = data["Item_Fat_Content"].replace({"LF": "Low Fat", "low fat": "Low Fat", "reg": "Regular"})

In [ ]:
data['Item_Visibility'].sort_values().unique()
#to check if Item_Visibility column have any 0 valuees or not

array([0.        , 0.0035747 , 0.0035891 , ..., 0.32363725, 0.32578081,
       0.32839095])

In [ ]:
# 2e. Fix Item_Visibility = 0 (items that exist can't have 0 visibility)
#Replace with mean visibility per item type
visibility_avg = data[data["Item_Visibility"] > 0].groupby("Item_Identifier")["Item_Visibility"].mean()
mask = data["Item_Visibility"] == 0
data.loc[mask, "Item_Visibility"] = data.loc[mask, "Item_Identifier"].map(visibility_avg)

#Item_Visibility is related to Item_identofoer as well. We can use the normal mean as well.
#But if we use the mean of each Item_identifier then the mean value will be better.
#Output : mean visibility per item type

In [ ]:
data["Item_Category"] = data["Item_Identifier"].str[:2].map({
    "FD": "Food", "NC": "Non-Consumable", "DR": "Drink"
})

In [ ]:
data.columns

Index(['Item_Identifier', 'Item_Weight', 'Item_Fat_Content', 'Item_Visibility',
       'Item_Type', 'Item_MRP', 'Outlet_Identifier',
       'Outlet_Establishment_Year', 'Outlet_Size', 'Outlet_Location_Type',
       'Outlet_Type', 'Item_Outlet_Sales', 'source', 'Item_Category'],
      dtype='object')

In [ ]:
# 2d. Outlet age
data["Outlet_Age"] = 2013 - data["Outlet_Establishment_Year"]

In [ ]:
# 2f. Visibility ratio: item visibility relative to average per item
data["Item_Visibility_MeanRatio"] = (data["Item_Visibility"] /data.groupby("Item_Identifier")["Item_Visibility"].transform("mean"))


In [ ]:
# 2g. MRP bins
data["Item_MRP_Cluster"] = pd.cut(
    data["Item_MRP"],
    bins=[0, 70, 130, 200, 300],
    labels=[0, 1, 2, 3]
).astype(int)

In [ ]:
data.isnull().sum()

,0
Item_Identifier,0
Item_Weight,2439
Item_Fat_Content,0
Item_Visibility,0
Item_Type,0
Item_MRP,0
Outlet_Identifier,0
Outlet_Establishment_Year,0
Outlet_Size,4016
Outlet_Location_Type,0


In [ ]:
data.groupby("Item_Identifier")["Item_Weight"].mean()

,Item_Weight
Item_Identifier,
DRA12,11.600
DRA24,19.350
DRA59,8.270
DRB01,7.390
DRB13,6.115
...,...
NCZ30,6.590
NCZ41,19.850
NCZ42,10.500


In [ ]:
data["Item_Weight"].fillna(data["Item_Weight"].mean(), inplace=True)

In [ ]:
outlet_size_mode = data.groupby("Outlet_Type")["Outlet_Size"].agg(
    lambda x: x.mode()[0] if not x.mode().empty else "Small"
)
mask = data["Outlet_Size"].isnull()
data.loc[mask, "Outlet_Size"] = data.loc[mask, "Outlet_Type"].map(outlet_size_mode)

print("Missing values after imputation:")
print(data[["Item_Weight", "Outlet_Size"]].isnull().sum())


Missing values after imputation:
Item_Weight    0
Outlet_Size    0
dtype: int64


In [ ]:
drop_cols = ["Item_Identifier", "Outlet_Identifier", "Outlet_Establishment_Year",
             "source", "Item_Outlet_Sales"]

cat_cols = ["Item_Fat_Content", "Item_Type", "Outlet_Size",
            "Outlet_Location_Type", "Outlet_Type", "Item_Category"]

le = LabelEncoder()
for col in cat_cols:
    data[col] = le.fit_transform(data[col].astype(str))

feature_cols = [c for c in data.columns if c not in drop_cols]
print(f"\nFeatures used ({len(feature_cols)}): {feature_cols}")



Features used (12): ['Item_Weight', 'Item_Fat_Content', 'Item_Visibility', 'Item_Type', 'Item_MRP', 'Outlet_Size', 'Outlet_Location_Type', 'Outlet_Type', 'Item_Category', 'Outlet_Age', 'Item_Visibility_MeanRatio', 'Item_MRP_Cluster']


In [ ]:
# 5. SPLIT BACK TO TRAIN/TEST
# ─────────────────────────────────────────────
train_data = data[data["source"] == "train"].copy()
test_data  = data[data["source"] == "test"].copy()

X_train = train_data[feature_cols].values
y_train = train_data["Item_Outlet_Sales"].values
X_test  = test_data[feature_cols].values

print(f"\nX_train: {X_train.shape}, X_test: {X_test.shape}")


X_train: (8523, 12), X_test: (5681, 12)


In [ ]:
# 6. MODEL TRAINING & CROSS-VALIDATION
# ─────────────────────────────────────────────

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# ── 6a. XGBoost ──────────────────────────────
xgb_params = {
    "n_estimators": 500,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 3,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "objective": "reg:squarederror",
    "random_state": 42,
    "n_jobs": -1,
}
xgb_model = xgb.XGBRegressor(**xgb_params)

xgb_oof  = np.zeros(len(X_train))
xgb_pred = np.zeros(len(X_test))
for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
    xgb_model.fit(
        X_train[tr_idx], y_train[tr_idx],
        eval_set=[(X_train[val_idx], y_train[val_idx])],
        verbose=False
    )
    xgb_oof[val_idx]  = xgb_model.predict(X_train[val_idx])
    xgb_pred         += xgb_model.predict(X_test) / kf.n_splits

xgb_rmse = np.sqrt(mean_squared_error(y_train, xgb_oof))
print(f"\nXGBoost CV RMSE: {xgb_rmse:.4f}")



XGBoost CV RMSE: 1137.8303


In [ ]:
# ── 6b. LightGBM ─────────────────────────────
lgb_params = {
    "n_estimators": 500,
    "max_depth": 6,
    "learning_rate": 0.05,
    "num_leaves": 50,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_samples": 20,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "objective": "regression",
    "random_state": 42,
    "n_jobs": -1,
    "verbose": -1,
}
lgb_model = lgb.LGBMRegressor(**lgb_params)

lgb_oof  = np.zeros(len(X_train))
lgb_pred = np.zeros(len(X_test))
for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
    lgb_model.fit(
        X_train[tr_idx], y_train[tr_idx],
        eval_set=[(X_train[val_idx], y_train[val_idx])],
    )
    lgb_oof[val_idx]  = lgb_model.predict(X_train[val_idx])
    lgb_pred         += lgb_model.predict(X_test) / kf.n_splits

lgb_rmse = np.sqrt(mean_squared_error(y_train, lgb_oof))
print(f"LightGBM CV RMSE: {lgb_rmse:.4f}")

LightGBM CV RMSE: 1121.6651


In [ ]:
# ── 6c. Stacking / Blending ───────────────────
# Use OOF predictions as meta-features for a Ridge blender
meta_train = np.column_stack([xgb_oof, lgb_oof])
meta_test  = np.column_stack([xgb_pred, lgb_pred])

ridge = Ridge(alpha=1.0)
ridge.fit(meta_train, y_train)
final_pred = ridge.predict(meta_test)

# Clip negatives (sales can't be negative)
final_pred = np.clip(final_pred, 0, None)

blend_rmse = np.sqrt(mean_squared_error(y_train, ridge.predict(meta_train)))
print(f"Blended  CV RMSE: {blend_rmse:.4f}")
print(f"\nRidge blend weights → XGB: {ridge.coef_[0]:.3f}, LGB: {ridge.coef_[1]:.3f}")


Blended  CV RMSE: 1117.5597

Ridge blend weights → XGB: 0.177, LGB: 0.762


In [ ]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 11.5 MB/s eta 0:00:00


In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── CELL 1: ADD TARGET ENCODING + EXTRA FEATURES ─────────────────
# Re-read identifiers (they were dropped earlier)
train_raw = pd.read_csv("/content/train_v9rqX0R.csv")
test_raw  = pd.read_csv("/content/test_AbJTz2l.csv")
train_raw["source"] = "train"; test_raw["source"] = "test"
data2 = pd.concat([train_raw, test_raw], ignore_index=True, sort=False)

In [ ]:
# Same cleaning as your notebook
data2.loc[data2["Item_Identifier"].str[:2] == "NC", "Item_Fat_Content"] = "Non-Edible"
data2["Item_Fat_Content"] = data2["Item_Fat_Content"].replace({"LF":"Low Fat","low fat":"Low Fat","reg":"Regular"})
vis_avg2 = data2[data2["Item_Visibility"] > 0].groupby("Item_Identifier")["Item_Visibility"].mean()
data2.loc[data2["Item_Visibility"]==0,"Item_Visibility"] = data2.loc[data2["Item_Visibility"]==0,"Item_Identifier"].map(vis_avg2)
data2["Item_Category"]             = data2["Item_Identifier"].str[:2].map({"FD":"Food","NC":"Non-Consumable","DR":"Drink"})
data2["Outlet_Age"]                = 2013 - data2["Outlet_Establishment_Year"]
data2["Item_Visibility_MeanRatio"] = data2["Item_Visibility"] / data2.groupby("Item_Identifier")["Item_Visibility"].transform("mean")
data2["Item_MRP_Cluster"]          = pd.cut(data2["Item_MRP"],bins=[0,70,130,200,300],labels=[0,1,2,3]).astype(int)
data2["Item_Weight"].fillna(data2["Item_Weight"].mean(), inplace=True)
mode_map2 = data2.groupby("Outlet_Type")["Outlet_Size"].agg(lambda x: x.mode()[0] if not x.mode().empty else "Small")
data2.loc[data2["Outlet_Size"].isnull(),"Outlet_Size"] = data2.loc[data2["Outlet_Size"].isnull(),"Outlet_Type"].map(mode_map2)

In [ ]:
# 3 extra engineered features
data2["Log_MRP"]          = np.log1p(data2["Item_MRP"])
data2["Visibility_x_MRP"] = data2["Item_Visibility"] * data2["Item_MRP"]
data2["MRP_per_Weight"]   = data2["Item_MRP"] / (data2["Item_Weight"] + 1e-5)

In [ ]:
# Target encoding — maps each category to its mean sales (train only, no leakage)
for col in ["Outlet_Identifier", "Item_Identifier", "Item_Type"]:
    gmap = train_raw.groupby(col)["Item_Outlet_Sales"].mean()
    data2[f"{col}_TargetEnc"] = data2[col].map(gmap).fillna(train_raw["Item_Outlet_Sales"].mean())

cat_cols = ["Item_Fat_Content", "Item_Type", "Outlet_Size", "Outlet_Location_Type", "Outlet_Type", "Item_Category"]
for col in cat_cols:
    data2[col] = LabelEncoder().fit_transform(data2[col].astype(str))

drop_cols2 = ["Item_Identifier","Outlet_Identifier","Outlet_Establishment_Year","source","Item_Outlet_Sales"]
feat2   = [c for c in data2.columns if c not in drop_cols2]
X2      = data2.iloc[:len(train_raw)][feat2].values
y2      = train_raw["Item_Outlet_Sales"].values
X2_test = data2.iloc[len(train_raw):][feat2].values

print(f"Features: {len(feat2)} → {feat2}")


Features: 18 → ['Item_Weight', 'Item_Fat_Content', 'Item_Visibility', 'Item_Type', 'Item_MRP', 'Outlet_Size', 'Outlet_Location_Type', 'Outlet_Type', 'Item_Category', 'Outlet_Age', 'Item_Visibility_MeanRatio', 'Item_MRP_Cluster', 'Log_MRP', 'Visibility_x_MRP', 'MRP_per_Weight', 'Outlet_Identifier_TargetEnc', 'Item_Identifier_TargetEnc', 'Item_Type_TargetEnc']


In [ ]:
# ── CELL 2: OPTUNA TUNING FOR XGBOOST ────────────────────────────
def xgb_objective(trial):
    p = dict(
        n_estimators     = trial.suggest_int('n_estimators', 300, 1000),
        max_depth        = trial.suggest_int('max_depth', 3, 8),
        learning_rate    = trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        subsample        = trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bytree = trial.suggest_float('colsample_bytree', 0.6, 1.0),
        min_child_weight = trial.suggest_int('min_child_weight', 1, 10),
        reg_alpha        = trial.suggest_float('reg_alpha', 1e-3, 10, log=True),
        reg_lambda       = trial.suggest_float('reg_lambda', 1e-3, 10, log=True),
        random_state=42, verbosity=0
    )
    oof = np.zeros(len(X2))
    for ti, vi in kf.split(X2):
        m = xgb.XGBRegressor(**p)
        m.fit(X2[ti], y2[ti])
        oof[vi] = m.predict(X2[vi])
    return np.sqrt(mean_squared_error(y2, oof))

xgb_study = optuna.create_study(direction='minimize')
xgb_study.optimize(xgb_objective, n_trials=25)
print(f"Best XGB RMSE: {xgb_study.best_value:.2f}")
print(f"Best XGB params: {xgb_study.best_params}")

Best XGB RMSE: 988.46
Best XGB params: {'n_estimators': 330, 'max_depth': 3, 'learning_rate': 0.020414123574365214, 'subsample': 0.7447437031288537, 'colsample_bytree': 0.9794167255369106, 'min_child_weight': 8, 'reg_alpha': 0.002697744742634553, 'reg_lambda': 1.0775307115792558}


In [ ]:
# ── CELL 3: OPTUNA TUNING FOR LIGHTGBM ───────────────────────────
def lgb_objective(trial):
    p = dict(
        n_estimators     = trial.suggest_int('n_estimators', 300, 1000),
        max_depth        = trial.suggest_int('max_depth', 3, 8),
        learning_rate    = trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        subsample        = trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bytree = trial.suggest_float('colsample_bytree', 0.6, 1.0),
        min_child_samples= trial.suggest_int('min_child_samples', 10, 50),
        reg_alpha        = trial.suggest_float('reg_alpha', 1e-3, 10, log=True),
        reg_lambda       = trial.suggest_float('reg_lambda', 1e-3, 10, log=True),
        random_state=42, verbose=-1
    )
    oof = np.zeros(len(X2))
    for ti, vi in kf.split(X2):
        m = lgb.LGBMRegressor(**p)
        m.fit(X2[ti], y2[ti])
        oof[vi] = m.predict(X2[vi])
    return np.sqrt(mean_squared_error(y2, oof))

lgb_study = optuna.create_study(direction='minimize')
lgb_study.optimize(lgb_objective, n_trials=25)
print(f"Best LGB RMSE: {lgb_study.best_value:.2f}")
print(f"Best LGB params: {lgb_study.best_params}")

# ── CELL 4: FINAL STACKED MODEL + SUBMISSION ─────────────────────
xgb_oof2=np.zeros(len(X2)); xgb_pred2=np.zeros(len(X2_test))
lgb_oof2=np.zeros(len(X2)); lgb_pred2=np.zeros(len(X2_test))

for ti, vi in kf.split(X2):
    m = xgb.XGBRegressor(**{**xgb_study.best_params, 'random_state':42, 'verbosity':0})
    m.fit(X2[ti], y2[ti])
    xgb_oof2[vi] = m.predict(X2[vi])
    xgb_pred2   += m.predict(X2_test) / 5

for ti, vi in kf.split(X2):
    m = lgb.LGBMRegressor(**{**lgb_study.best_params, 'random_state':42, 'verbose':-1})
    m.fit(X2[ti], y2[ti])
    lgb_oof2[vi] = m.predict(X2[vi])
    lgb_pred2   += m.predict(X2_test) / 5

ridge2 = Ridge()
ridge2.fit(np.column_stack([xgb_oof2, lgb_oof2]), y2)
final_pred2 = np.clip(ridge2.predict(np.column_stack([xgb_pred2, lgb_pred2])), 0, None)
stack_rmse  = np.sqrt(mean_squared_error(y2, ridge2.predict(np.column_stack([xgb_oof2, lgb_oof2]))))

print(f"\n{'='*45}")
print(f"Your original blend RMSE  : 1117.91")
print(f"Optimized stack RMSE      : {stack_rmse:.2f}")
print(f"Improvement               : {1117.91 - stack_rmse:.2f} points ↓")
print(f"Ridge weights  XGB={ridge2.coef_[0]:.3f}  LGB={ridge2.coef_[1]:.3f}")
print(f"{'='*45}")

Best LGB RMSE: 987.85
Best LGB params: {'n_estimators': 376, 'max_depth': 4, 'learning_rate': 0.012536762849735221, 'subsample': 0.9406315516095306, 'colsample_bytree': 0.7970099826988966, 'min_child_samples': 29, 'reg_alpha': 0.7623385648919331, 'reg_lambda': 0.3760722319175669}

Your original blend RMSE  : 1117.91
Optimized stack RMSE      : 987.12
Improvement               : 130.79 points ↓
Ridge weights  XGB=0.380  LGB=0.633


In [ ]:
submission = pd.DataFrame({
    'Item_Identifier':  test_raw['Item_Identifier'],
    'Outlet_Identifier': test_raw['Outlet_Identifier'],
    'Item_Outlet_Sales': final_pred2
})
submission.to_csv("submission_optimized.csv", index=False)
print("Saved: submission_optimized.csv")

Saved: submission_optimized.csv


In [ ]:
from google.colab import files
files.download("submission_optimized.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>